In [19]:
import sys, os
sys.path.append(os.path.abspath(".."))
from scraping.scraper import crawler, create_session


In [ ]:
session=create_session()
documents = crawler("https://www.learnpytorch.io/", session, initial_referer="https://www.google.com/")

Crawling: https://www.learnpytorch.io/
Crawling: https://www.learnpytorch.io/pytorch_most_common_errors/
Crawling: https://www.learnpytorch.io/07_pytorch_experiment_tracking/
Crawling: https://www.learnpytorch.io/03_pytorch_computer_vision/
Crawling: https://www.learnpytorch.io/pytorch_cheatsheet/
Crawling: https://www.learnpytorch.io/02_pytorch_classification/
Crawling: https://www.learnpytorch.io/09_pytorch_model_deployment/
Crawling: https://www.learnpytorch.io/05_pytorch_going_modular/
Crawling: https://www.learnpytorch.io/08_pytorch_paper_replicating/
Crawling: https://www.learnpytorch.io/01_pytorch_workflow/
Crawling: https://www.learnpytorch.io/04_pytorch_custom_datasets/
Crawling: https://www.learnpytorch.io/pytorch_2_intro/
Crawling: https://www.learnpytorch.io/pytorch_extra_resources/
Crawling: https://www.learnpytorch.io/06_pytorch_transfer_learning/
Crawling: https://www.learnpytorch.io/00_pytorch_fundamentals/
Crawling: https://www.learnpytorch.io/08_pytorch_paper_replicat

In [21]:
# documents

In [23]:
from database.vector_storage import save_vault, load_vault

from chunking.fixed_chunker import fixed_chunk_documents
from chunking.recursive_chunker import recursive_chunk_documents
from chunking.semantic_chunker import semantic_chunk_documents
from chunking.code_aware_chunker import code_aware_chunk_documents
from chunking.header_chunker import header_aware_chunk_documents 
from chunking.sliding_chunker import sliding_chunk_documents
from embeddings.embedder import embed_documents,load_embedding_model,embed_query

model = load_embedding_model()


[Init] Waking up RTX 5060 & loading BAAI/bge-base-en-v1.5...
[Init] GPU Model locked to CUDAExecutionProvider in 0.50s


In [ ]:


VAULT_DIR = r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\local_vector_vault"

# 1. Try to load from disk first
my_vault = load_vault(VAULT_DIR)

# 2. If it doesn't exist, calculate it on the GPU, then save it
if my_vault is None:
    print("No local vault found. Generating embeddings on GPU...")
    
    fixed_chunks = fixed_chunk_documents(documents, chunk_size=512)
    header_chunks = header_aware_chunk_documents(documents)
    sliding_chunks = sliding_chunk_documents(documents, chunk_size=512, overlap=128)
    recursive_chunks = recursive_chunk_documents(documents, chunk_size=512, overlap=128)
    code_chunks = code_aware_chunk_documents(documents, max_chars=512, overlap=128)
    semantic_chunks = semantic_chunk_documents(documents, model)

    header_embeddings = embed_documents(header_chunks,model)
    sliding_embeddings = embed_documents(sliding_chunks,model)
    recursive_embeddings = embed_documents(recursive_chunks,model)
    code_embeddings = embed_documents(code_chunks,model)
    semantic_embeddings = embed_documents(semantic_chunks,model)
    fixed_embeddings = embed_documents(fixed_chunks,model)
    
    my_vault = {
        "Fixed Size": {"chunks": fixed_chunks, "matrix": fixed_embeddings},
        "Sliding window": {"chunks": sliding_chunks, "matrix": sliding_embeddings},
        "Recursive": {"chunks": recursive_chunks, "matrix": recursive_embeddings},
        "Header Aware": {"chunks": header_chunks, "matrix": header_embeddings},
        "Code Aware Sliding": {"chunks": sliding_chunks, "matrix": sliding_embeddings},
        "Semantic": {"chunks": semantic_chunks, "matrix": semantic_embeddings}
    }
    
    # Save it to disk permanently
    save_vault(my_vault, VAULT_DIR)

# 3. Proceed directly to RRF Fusion and Battle Arena!
# run_rrf_fusion("How to check device?", model, my_vault)

# With this in place, your workflow becomes completely decoupled:
# 1. **Kernel Restart:** Takes 0 seconds.
# 2. **Reloading Embeddings:** Takes `~0.3` seconds (NumPy binary loads are blazingly fast).
# 3. **Experimenting with RRF:** You can edit your Reciprocal Rank Fusion formulas, test new queries, and analyze the outputs instantly without ever waking up the RTX 5060 to re-embed the text.

⚠️ No local vault found. Generating embeddings on GPU...


GPU Vectorizing (1737 chunks): 100%|██████████| 1737/1737 [00:05<00:00, 327.59chunk/s]

Vault securely persisted to disk at: C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\notebooks\local_vector_vault


In [6]:
query = "Expected all tensors to be on the same device, but found at least two devices"
query_embedding = embed_query(query, model)


In [ ]:
# from retrieval.similarity import get_top_k_similar_documents
# top_k_header = get_top_k_similar_documents(query_embedding, header_embeddings, header_chunks, k=3)
# top_k_sliding = get_top_k_similar_documents(query_embedding, sliding_embeddings, sliding_chunks, k=3)
# top_k_recursive = get_top_k_similar_documents(query_embedding, recursive_embeddings, recursive_chunks, k=3)
# top_k_semantic = get_top_k_similar_documents(query_embedding, semantic_embeddings, semantic_chunks, k=3)

In [ ]:

from evaluation.retrieval_evaluator import benchmark_chunking_strategies,execute_rrf_fusion,run_stress_test_suite

benchmark_chunking_strategies(
    "How do I put a tensor on the GPU?", model, my_vault, top_k=2
)

🔍 QUERY: 「 How do I put a tensor on the GPU? 」

📦 STRATEGY: 【 Header Aware 】
  #1 [Sim: 0.8374] ── [00_pytorch_fundamentals > Set device type > 3. Putting tensors (and models) on the GPU¶]
      "You can put tensors (and models, we'll see this later) on a specific device by callingto(device)on them. Wheredeviceis the target device you'd like the tensor (or model) to go to. ..."

  #2 [Sim: 0.8063] ── [00_pytorch_fundamentals > Move tensor to GPU (if available)]
      "tensor_on_gpu = tensor.to(device) ⏎ tensor_on_gpu ⏎ ``` ⏎  ⏎ ``` ⏎ tensor([1, 2, 3]) cpu ⏎ ``` ⏎  ⏎ ``` ⏎ tensor([1, 2, 3], device='mps:0') ⏎ ``` ⏎  ⏎ If you have a GPU available, ..."


📦 STRATEGY: 【 Sliding Window 】
  #1 [Sim: 0.8296] ── [00_pytorch_fundamentals > Set device type > 3. Putting tensors (and models) on the GPU¶]
      "o(device)on them. Wheredeviceis the target device you'd like the tensor (or model) to go to. ⏎  ⏎ Why do this? ⏎  ⏎ GPUs offer far faster numerical computing than CPUs do and if a ..."

  #2

In [ ]:
test_queries = [
    # Group 1: Syntax
    "What is the exact code to manually set the random seed for CPU and CUDA?",
    "Show me the code to check which device a model's parameters are sitting on.",
    # Group 2: Semantic Gaslight
    "Why am I getting the RuntimeError: Expected all tensors to be on the same device?",
    "How do I fix a shape mismatch error inside nn.Linear forward pass?",
    # Group 3: Needle in the Haystack
    "Does the Zero to Mastery PyTorch course cover PyTorch version 2.0?",
    "What specific data science bootcamp is recommended as a prerequisite before taking this course?",
    # Group 4: Conceptual
    "What is the overarching computer vision project built throughout the milestone chapters called?",
    "What is the difference between torch.rand and torch.randn?",
]

execute_rrf_fusion(test_queries, my_vault, model, top_k=3)

🏁 OVERALL SERIES CHAMPIONSHIP:
  Fixed Size: 1 / 8 rounds won
  Sliding window: 1 / 8 rounds won
  Recursive: 2 / 8 rounds won
  Header Aware: 3 / 8 rounds won
  Code Aware Sliding: 0 / 8 rounds won
  Semantic: 1 / 8 rounds won


In [ ]:
run_stress_test_suite(test_queries, my_vault, model, top_k=3)

🧬 FUSED RETRIEVAL FOR: 「 Show me the code to check which device a model is sitting on. 」
 #1 [RRF Score: 0.05852] ── 03_pytorch_computer_vision::Move values to device
     "torch.manual_seed(42) ⏎ def eval_model(model: torch.nn.Module,  ⏎                data_loader: torch.utils.data.DataLoader,  ⏎                loss_fn: torch.nn.M..."

 #2 [RRF Score: 0.03080] ── 01_pytorch_workflow::Put model to target device (if your data is on GPU, model will have to be on GPU to make predictions)
     "loaded_model_1.to(device) ⏎  ⏎ print(f"Loaded model:\n{loaded_model_1}") ⏎ print(f"Model on device:\n{next(loaded_model_1.parameters()).device}") ⏎ ``` ⏎  ⏎ ```..."

 #3 [RRF Score: 0.02944] ── 03_pytorch_computer_vision::Calculate model 1 results with device-agnostic code
     "model_1_results = eval_model(model=model_1, data_loader=test_dataloader, ⏎     loss_fn=loss_fn, accuracy_fn=accuracy_fn, ⏎     device=device ⏎ ) ⏎ model_1_resul..."

